In [20]:
import ioh
from Practical_Problems import get_practical_problem
import numpy as np
import os

In [21]:
triggers = [ioh.logger.trigger.OnImprovement(), ioh.logger.trigger.Each(10)]

logger = ioh.logger.Analyzer(
    triggers=triggers,
    root=os.path.curdir,
    algorithm_name="RandomSearch",
    folder_name="Results-RS",
    algorithm_info="A simple random search optimizer",
    store_positions=False
)

In [22]:
# Set the dimensionality
dimension:int = 26

# Try to get the Tersoff_potentialC1
problem_instance = get_practical_problem(1126,dim=dimension)
print(problem_instance.meta_data.name)

Spacecraft_Trajectory_OptimizationC1


In [23]:
# Test attaching the ioh logger
problem_instance.attach_logger(logger)

In [24]:
class RandomSearchOptimizer:
    def __init__(self, problem:callable, n_iterations=1000, lower_bound=-5, upper_bound=5):

        if not callable(problem):
            raise ValueError("The problem must be a callable function.")
        
        self.problem = problem
        self.n_iterations = n_iterations
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound
        self.best_solution = None
        self.best_objective = float('inf')

    def optimize(self, **kwargs):

        random_seed = kwargs.get('random_seed', 43)

        if isinstance(random_seed, int):
            rng = np.random.default_rng(random_seed)
        else:
            raise ValueError("random_seed must be an integer.")

        for ii in range(self.n_iterations):
            candidate = rng.uniform(self.lower_bound, self.upper_bound, self.problem.meta_data.n_variables)
            objective_value = self.problem(candidate)
            if objective_value < self.best_objective:
                self.best_objective = objective_value
                self.best_solution = candidate
                print(f"New best solution at iteration {ii}: objective {self.best_objective}")
        
        self.problem.reset()
        return self.best_solution, self.best_objective
    
    def reset(self):
        self.best_solution = None
        self.best_objective = float('inf')

In [25]:
optimizer = RandomSearchOptimizer(problem_instance, n_iterations=1000, lower_bound=-5, upper_bound=5)
# Run a loop of optimizations
n_runs = 5

for run in range(n_runs):
    print(f"Run {run+1} out of {n_runs}")
    best_solution, best_objective = optimizer.optimize(random_seed=run+50)
    print(f"Best solution: {best_solution}, Best objective: {best_objective}")
    optimizer.reset()

# Detach the logger
problem_instance.detach_logger()

Run 1 out of 5
New best solution at iteration 0: objective 99.4035196310005
New best solution at iteration 2: objective 81.56568972731705
New best solution at iteration 7: objective 67.80057426154568
New best solution at iteration 38: objective 66.1313535449346
New best solution at iteration 132: objective 54.50383553987619
New best solution at iteration 219: objective 51.087867897014156
New best solution at iteration 258: objective 44.89510751024286
New best solution at iteration 494: objective 41.948577424021856
Best solution: [ 1.36909417  0.645845   -3.80889885 -0.05486741  3.31054111 -1.16120464
 -2.45680469 -0.58539594  2.68060712  1.5346516  -1.94120695  2.81741097
 -4.22646612 -0.26532116 -4.69310641 -2.36456828 -2.64919036  1.36771353
 -2.90548735 -0.16737088  1.8289939  -3.8087346   3.90998839  4.98672367
  2.87729533  1.73379979], Best objective: 41.948577424021856
Run 2 out of 5
New best solution at iteration 0: objective 110.88643973654982
New best solution at iteration 1: